# 🏆 Générateur de Dataset Fake pour K-Means - StreetLeague

### Objectif : Créer une dataset réaliste pour segmenter les communautés
### Configuration : 1000 communautés par défaut

## 📋 Instructions d'utilisation
1. **Importer ce notebook** dans Google Colab
2. **Exécuter toutes les cellules** (Runtime > Run all)
3. **Télécharger les fichiers** générés automatiquement
4. **Utiliser les fichiers** dans votre projet Spring Boot

In [ ]:
# Importations nécessaires
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import random
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

# Configuration
np.random.seed(42)
random.seed(42)

print("📦 Bibliothèques importées avec succès!")
print(f"📅 Date d'exécution : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 🎯 Configuration du nombre de communautés

**1000 communautés** - Configuration optimale pour commencer

In [ ]:
# NOMBRE DE COMMUNAUTÉS À GÉNÉRER
# 1000 = Configuration optimale recommandée
NOMBRE_COMMUNAUTES = 1000  # <- Parfait pour commencer !

print(f"🎯 Génération de {NOMBRE_COMMUNAUTES} communautés...")
print(f"📊 Temps d'exécution estimé : 1-2 minutes")
print(f"🎯 Score de silhouette attendu : 0.75-0.85")
print(f"📈 Distribution attendue : ~200 actives, ~600 modérées, ~200 endormies")

## 🏗️ Génération des données de base

Création de 3 clusters naturels : ultra-actives, modérées, endormies

In [ ]:
# Génération des communautés avec 3 clusters naturels
def generate_communities(n_communities):
    communities = []
    
    for i in range(n_communities):
        # Déterminer le type de communauté (3 clusters naturels)
        cluster_type = np.random.choice(['active', 'moderate', 'dormant'], p=[0.2, 0.6, 0.2])
        
        if cluster_type == 'active':
            # Communautés ultra-actives (20%)
            total_events = np.random.randint(50, 200)
            events_last_30 = np.random.randint(8, 25)
            events_last_90 = np.random.randint(25, 80)
            unique_organizers = np.random.randint(8, 25)
            community_age = np.random.randint(180, 730)  # 6 mois à 2 ans
            last_event_days = np.random.randint(0, 7)
            community_type = np.random.choice(['sportive', 'culturelle', 'sociale'], p=[0.6, 0.2, 0.2])
            
        elif cluster_type == 'moderate':
            # Communautés modérées (60%)
            total_events = np.random.randint(10, 50)
            events_last_30 = np.random.randint(2, 8)
            events_last_90 = np.random.randint(8, 25)
            unique_organizers = np.random.randint(2, 8)
            community_age = np.random.randint(90, 365)  # 3 mois à 1 an
            last_event_days = np.random.randint(7, 30)
            community_type = np.random.choice(['sportive', 'culturelle', 'sociale'], p=[0.4, 0.3, 0.3])
            
        else:  # dormant
            # Communautés endormies (20%)
            total_events = np.random.randint(1, 10)
            events_last_30 = np.random.randint(0, 2)
            events_last_90 = np.random.randint(0, 8)
            unique_organizers = np.random.randint(1, 3)
            community_age = np.random.randint(30, 180)  # 1 mois à 6 mois
            last_event_days = np.random.randint(30, 90)
            community_type = np.random.choice(['sportive', 'culturelle', 'sociale'], p=[0.3, 0.4, 0.3])
        
        # Calcul des features dérivées
        avg_events_per_month = total_events / max(community_age / 30, 1)
        diversity_score = unique_organizers / max(total_events, 1)
        activity_level = (events_last_30 * 3 + events_last_90) / 4  # Pondération récente
        
        community = {
            'community_id': i + 1,
            'community_name': f'Community_{i+1:04d}',
            'community_type': community_type,
            'creation_date_days_ago': community_age,
            'total_events': total_events,
            'events_last_30_days': events_last_30,
            'events_last_90_days': events_last_90,
            'unique_organizers': unique_organizers,
            'avg_events_per_month': round(avg_events_per_month, 2),
            'community_age_days': community_age,
            'last_event_days_ago': last_event_days,
            'diversity_score': round(diversity_score, 3),
            'activity_level': round(activity_level, 2),
            'true_cluster': cluster_type  # Pour validation
        }
        
        communities.append(community)
    
    return pd.DataFrame(communities)

# Génération du dataset
print("🏗️ Génération en cours...")
df = generate_communities(NOMBRE_COMMUNAUTES)

print(f"✅ Dataset généré : {len(df)} communautés")
print(f"📊 Colonnes : {list(df.columns)}")
print(f"🎯 Distribution attendue : ~200 actives, ~600 modérées, ~200 endormies")

# Afficher les premières lignes
print("\n📋 Aperçu des données :")
df.head(10)

## 📊 Analyse exploratoire des données

In [ ]:
# Statistiques descriptives
print("📈 Statistiques descriptives :")
print(df.describe())

# Distribution des vrais clusters
print("\n🎯 Distribution des clusters naturels :")
cluster_dist = df['true_cluster'].value_counts()
print(cluster_dist)
print(f"\n📊 Pourcentages :")
for cluster, count in cluster_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  - {cluster}: {count} communautés ({percentage:.1f}%)")

# Distribution des types de communauté
print("\n🏷️ Distribution des types :")
type_dist = df['community_type'].value_counts()
print(type_dist)

# Vérification des valeurs manquantes
print("\n🔍 Valeurs manquantes :")
print(df.isnull().sum())

## 📈 Visualisation des clusters naturels

In [ ]:
# Configuration des graphiques
plt.style.use('seaborn-v0_8')
plt.figure(figsize=(15, 10))

# Scatter plot 1: Events vs Activity Level
plt.subplot(2, 2, 1)
sns.scatterplot(data=df, x='total_events', y='activity_level', hue='true_cluster', palette='viridis', alpha=0.7)
plt.title('Total Events vs Activity Level')
plt.xlabel('Total Events')
plt.ylabel('Activity Level')
plt.legend(title='True Cluster')

# Scatter plot 2: Age vs Events
plt.subplot(2, 2, 2)
sns.scatterplot(data=df, x='community_age_days', y='events_last_30_days', hue='true_cluster', palette='viridis', alpha=0.7)
plt.title('Community Age vs Recent Events')
plt.xlabel('Community Age (days)')
plt.ylabel('Events Last 30 Days')
plt.legend(title='True Cluster')

# Box plot: Activity by cluster
plt.subplot(2, 2, 3)
sns.boxplot(data=df, x='true_cluster', y='activity_level', palette='Set2')
plt.title('Activity Level by True Cluster')
plt.xlabel('Cluster Type')
plt.ylabel('Activity Level')

# Count plot: Cluster distribution
plt.subplot(2, 2, 4)
sns.countplot(data=df, x='true_cluster', palette='Set2')
plt.title('Cluster Distribution')
plt.xlabel('Cluster Type')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

print("📊 Graphiques générés avec succès !")

## 🤖 Préparation pour K-Means

In [ ]:
# Sélection des features numériques
numeric_features = [
    'total_events', 'events_last_30_days', 'events_last_90_days',
    'unique_organizers', 'avg_events_per_month', 'community_age_days',
    'last_event_days_ago', 'diversity_score', 'activity_level'
]

# Encodage des variables catégorielles
le = LabelEncoder()
df['community_type_encoded'] = le.fit_transform(df['community_type'])

# Features finales pour le clustering
features_for_clustering = numeric_features + ['community_type_encoded']
X = df[features_for_clustering]

# Standardisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"🔧 Features pour clustering : {len(features_for_clustering)}")
print(f"📏 Shape des données : {X_scaled.shape}")
print(f"✨ Features : {features_for_clustering}")
print(f"📐 Moyenne après standardisation : {X_scaled.mean():.6f}")
print(f"📐 Écart-type après standardisation : {X_scaled.std():.6f}")

# Afficher les classes encodées
print(f"\n🏷️ Encodage des types de communauté :")
for i, type_name in enumerate(le.classes_):
    print(f"  {type_name} → {i}")

## 🎯 Application de K-Means

Application de l'algorithme avec 3 clusters

In [ ]:
print("🤖 Application de K-Means...")

# Application de K-Means avec 3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# Ajout des labels au dataframe
df['predicted_cluster'] = cluster_labels
df['predicted_cluster_name'] = df['predicted_cluster'].map({
    0: 'CLUSTER_0',
    1: 'CLUSTER_1', 
    2: 'CLUSTER_2'
})

# Calcul du score de silhouette
silhouette_avg = silhouette_score(X_scaled, cluster_labels)

print(f"🎯 K-Means appliqué avec succès !")
print(f"📊 Score de silhouette : {silhouette_avg:.3f}")
print(f"🏷️ Labels prédits : {np.unique(cluster_labels)}")

# Affichage des centres des clusters
print("\n🎯 Centres des clusters :")
centers = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers, columns=features_for_clustering)
print(centers_df.round(2))

# Interprétation du score de silhouette
if silhouette_avg > 0.7:
    print("\n✅ Score de silhouette EXCELLENT (>0.7)")
elif silhouette_avg > 0.5:
    print("\n👍 Score de silhouette BON (0.5-0.7)")
else:
    print("\n⚠️ Score de silhouette FAIBLE (<0.5)")

# Distribution des clusters prédits
print("\n📊 Distribution des clusters prédits :")
predicted_dist = df['predicted_cluster_name'].value_counts()
for cluster, count in predicted_dist.items():
    percentage = (count / len(df)) * 100
    print(f"  - {cluster}: {count} communautés ({percentage:.1f}%)")

## 📊 Visualisation des résultats

In [ ]:
# Visualisation des résultats du clustering
plt.figure(figsize=(15, 10))

# Comparaison : Vrais clusters vs Prédits
plt.subplot(2, 2, 1)
sns.scatterplot(data=df, x='total_events', y='activity_level', hue='true_cluster', palette='viridis', alpha=0.7)
plt.title('Vrais Clusters')
plt.xlabel('Total Events')
plt.ylabel('Activity Level')
plt.legend(title='True Cluster')

plt.subplot(2, 2, 2)
sns.scatterplot(data=df, x='total_events', y='activity_level', hue='predicted_cluster_name', palette='Set2', alpha=0.7)
plt.title('Clusters Prédits par K-Means')
plt.xlabel('Total Events')
plt.ylabel('Activity Level')
plt.legend(title='Predicted Cluster')

# Distribution des clusters prédits
plt.subplot(2, 2, 3)
sns.countplot(data=df, x='predicted_cluster_name', palette='Set2')
plt.title('Distribution des Clusters Prédits')
plt.xlabel('Cluster Prédit')
plt.ylabel('Nombre de Communautés')

# Matrice de confusion
plt.subplot(2, 2, 4)
confusion_matrix = pd.crosstab(df['true_cluster'], df['predicted_cluster_name'])
sns.heatmap(confusion_matrix, annot=True, cmap='Blues', fmt='d')
plt.title('Matrice de Confusion')
plt.xlabel('Cluster Prédit')
plt.ylabel('Vrai Cluster')

plt.tight_layout()
plt.show()

print("📊 Visualisations complétées !")

## 📋 Analyse détaillée des clusters

In [ ]:
# Analyse détaillée des clusters
def analyze_clusters(df):
    cluster_analysis = df.groupby('predicted_cluster_name').agg({
        'community_id': 'count',
        'total_events': ['mean', 'std'],
        'events_last_30_days': ['mean', 'std'],
        'unique_organizers': ['mean', 'std'],
        'activity_level': ['mean', 'std'],
        'community_age_days': ['mean', 'std']
    }).round(2)
    
    cluster_analysis.columns = ['_'.join(col).strip() for col in cluster_analysis.columns]
    return cluster_analysis

cluster_stats = analyze_clusters(df)
print("📊 Analyse statistique des clusters :")
print(cluster_stats)

# Interprétation des clusters
print("\n🎯 Interprétation des clusters :")
for cluster in df['predicted_cluster_name'].unique():
    cluster_data = df[df['predicted_cluster_name'] == cluster]
    avg_events = cluster_data['events_last_30_days'].mean()
    avg_organizers = cluster_data['unique_organizers'].mean()
    avg_activity = cluster_data['activity_level'].mean()
    
    if avg_events > 8 and avg_organizers > 6:
        cluster_type = "🔥 COMMUNAUTÉS ULTRA-ACTIVES"
        action = "Marketing premium, support prioritaire"
    elif avg_events > 3 and avg_organizers > 2:
        cluster_type = "⚡ COMMUNAUTÉS MODÉRÉES"
        action = "Support standard, encouragement"
    else:
        cluster_type = "😴 COMMUNAUTÉS ENDORMIES"
        action = "Relance prioritaire, aide active"
    
    print(f"\n{cluster} : {cluster_type}")
    print(f"  - Moyenne événements/30j : {avg_events:.1f}")
    print(f"  - Moyenne organisateurs : {avg_organizers:.1f}")
    print(f"  - Niveau d'activité : {avg_activity:.1f}")
    print(f"  - Nombre de communautés : {len(cluster_data)}")
    print(f"  - Action recommandée : {action}")

# Exemples de communautés par cluster
print("\n📝 Exemples de communautés par cluster :")
for cluster in df['predicted_cluster_name'].unique():
    examples = df[df['predicted_cluster_name'] == cluster].head(3)
    print(f"\n{cluster} :")
    for _, row in examples.iterrows():
        print(f"  - {row['community_name']}: {row['total_events']} événements, activité {row['activity_level']}")

## 💾 Export du dataset pour votre projet

In [ ]:
# Export du dataset complet
output_filename = f'streetleague_communities_dataset_{NOMBRE_COMMUNAUTES}_rows.csv'

df.to_csv(output_filename, index=False)
print(f"💾 Dataset complet exporté : {output_filename}")
print(f"📊 Taille : {len(df)} lignes × {len(df.columns)} colonnes")

# Export des features pour le modèle (prêt pour Spring Boot)
features_filename = f'streetleague_features_{NOMBRE_COMMUNAUTES}_rows.csv'
X_with_clusters = X.copy()
X_with_clusters['predicted_cluster'] = cluster_labels
X_with_clusters['community_id'] = df['community_id']
X_with_clusters['community_name'] = df['community_name']
X_with_clusters.to_csv(features_filename, index=False)
print(f"🔧 Features pour modèle exportées : {features_filename}")
print(f"📏 Taille features : {len(X_with_clusters)} lignes × {len(X_with_clusters.columns)} colonnes")

# Export du modèle entraîné
model_filename = f'kmeans_model_{NOMBRE_COMMUNAUTES}_rows.pkl'
joblib.dump(kmeans, model_filename)
print(f"🤖 Modèle K-Means exporté : {model_filename}")

# Export du scaler
scaler_filename = f'scaler_{NOMBRE_COMMUNAUTES}_rows.pkl'
joblib.dump(scaler, scaler_filename)
print(f"📐 Scaler exporté : {scaler_filename}")

# Export du label encoder
encoder_filename = f'label_encoder_{NOMBRE_COMMUNAUTES}_rows.pkl'
joblib.dump(le, encoder_filename)
print(f"🏷️ Label encoder exporté : {encoder_filename}")

# Export des métadonnées
metadata = {
    'n_communities': NOMBRE_COMMUNAUTES,
    'n_clusters': 3,
    'silhouette_score': silhouette_avg,
    'features': features_for_clustering,
    'cluster_labels': {i: f'CLUSTER_{i}' for i in range(3)},
    'creation_date': datetime.now().isoformat()
}

import json
metadata_filename = f'metadata_{NOMBRE_COMMUNAUTES}_rows.json'
with open(metadata_filename, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"📋 Métadonnées exportées : {metadata_filename}")

print("\n✅ Tous les fichiers prêts pour votre projet Spring Boot !")

# Résumé des fichiers
files_created = [
    output_filename,
    features_filename,
    model_filename,
    scaler_filename,
    encoder_filename,
    metadata_filename
]

print(f"\n📂 Fichiers créés ({len(files_created)}) :")
for file in files_created:
    print(f"  - {file}")

## 📥 Téléchargement des fichiers

Téléchargement automatique de tous les fichiers générés

In [ ]:
# Téléchargement automatique des fichiers
try:
    from google.colab import files
    
    print("📥 Démarrage du téléchargement...")
    
    # Téléchargement de tous les fichiers
    for file_path in files_created:
        try:
            files.download(file_path)
            print(f"✅ {file_path} téléchargé")
        except Exception as e:
            print(f"⚠️ Erreur lors du téléchargement de {file_path}: {e}")
    
    print("\n🎉 Tous les fichiers sont téléchargés !")
    print("📂 Vous pouvez maintenant les importer dans votre projet Spring Boot")
    
except ImportError:
    print("⚠️ Vous n'êtes pas sur Google Colab")
    print("📂 Les fichiers sont sauvegardés localement dans le répertoire de travail")
    print("🔍 Vous pouvez les trouver dans l'explorateur de fichiers de Colab")
    
    # Lister les fichiers dans le répertoire courant
    import os
    current_files = [f for f in os.listdir('.') if f.endswith(('.csv', '.pkl', '.json'))]
    print(f"\n📋 Fichiers disponibles :")
    for file in sorted(current_files):
        size = os.path.getsize(file) / 1024  # taille en KB
        print(f"  - {file} ({size:.1f} KB)")

## 🎯 Instructions pour utiliser ces fichiers dans votre projet

### 1. **Dataset complet** (`streetleague_communities_dataset_1000_rows.csv`)
- Contient toutes les données avec clusters prédits
- Utilisez-le pour tester votre API Spring Boot
- Colonnes : community_id, community_name, predicted_cluster, etc.

### 2. **Features pour modèle** (`streetleague_features_1000_rows.csv`)
- Contient uniquement les features numériques
- Prêt pour être utilisé dans votre service ML
- Colonnes : total_events, events_last_30_days, etc.

### 3. **Modèle K-Means** (`kmeans_model_1000_rows.pkl`)
- Modèle entraîné avec 1000 communautés
- Chargez-le dans votre service Python/FastAPI
- `joblib.load('kmeans_model_1000_rows.pkl')`

### 4. **Scaler** (`scaler_1000_rows.pkl`)
- Standardisez les nouvelles données avec le même scaler
- `joblib.load('scaler_1000_rows.pkl')`
- Essentiel pour des prédictions cohérentes

### 5. **Label Encoder** (`label_encoder_1000_rows.pkl`)
- Encodez les types de communauté de manière cohérente
- `joblib.load('label_encoder_1000_rows.pkl')`
- Maintient la correspondance des types

### 6. **Métadonnées** (`metadata_1000_rows.json`)
- Informations sur le modèle et les données
- Score de silhouette, features utilisées, etc.
- Utile pour la documentation et le versioning

### 7. **Intégration Spring Boot**
```java
@RestController
@RequestMapping("/api/ml")
public class CommunitySegmentationController {
    
    @GetMapping("/segment-communities")
    public List<CommunityCluster> segmentCommunities() {
        return mlService.segmenterCommunautés();
    }
    
    @PostMapping("/predict-cluster")
    public ClusterResponse predictCluster(@RequestBody CommunityRequest request) {
        return mlService.predictCluster(request);
    }
}
```

### 8. **Service Python/FastAPI**
```python
import joblib
import pandas as pd
from sklearn.preprocessing import StandardScaler

# Charger le modèle et les outils
model = joblib.load('kmeans_model_1000_rows.pkl')
scaler = joblib.load('scaler_1000_rows.pkl')
encoder = joblib.load('label_encoder_1000_rows.pkl')

def predict_cluster(features):
    # Prétraitement
    features_scaled = scaler.transform(features)
    # Prédiction
    cluster = model.predict(features_scaled)
    return cluster[0]
```

### 9. **Prochaines étapes**
1. Importez les CSV dans votre base de données
2. Créez le service ML avec Python/FastAPI
3. Connectez Spring Boot au service ML
4. Développez l'interface frontend
5. Testez avec les 1000 communautés générées

### 10. **Monitoring et maintenance**
- Surveillez le score de silhouette
- Ré-entraînez le modèle avec de nouvelles données
- Versionnez vos modèles avec les métadonnées
- Testez les prédictions sur de nouvelles communautés

🚀 **Votre dataset de 1000 communautés est prêt !**

---

## 📞 Support

Si vous avez des questions :
- Vérifiez les métadonnées pour comprendre le modèle
- Utilisez les graphiques pour visualiser les clusters
- Testez avec quelques exemples avant de déployer

**Bon développement avec votre modèle de segmentation !**